In [2]:
## Run this cell if you are running your notebook in Google Colab!
## Make sure the data for this lesson is in the folder 'Digital Transformation Notebooks/Data'
## OR, change the location to the appropriate folder in your drive

## mount google colab
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/My Drive/Digital Transformation Notebooks/Data

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content/gdrive/My Drive/Digital Transformation Notebooks/Data


# Solving the Energy Systems Unit Commitment Problem in Pyomo

---



In our last end of module notebook, we looked at the Energy Systems Economic Dispatch problem in Pyomo, and solved it as a linear optimization problem. We saw that the ability to turn off generators operating with a minimum output at the optimal solution could possibly improve our costs, but this would require binary variables. We now have the tools to do this in Pyomo.

### The Economic Dispatch Problem

Recall that the economic dispatch problem minmized the cost of meeting energy demands by the electrical grid subject to physical constraints on various power source systems (e.g., wind turbines, thermal generation, etc.)

To summarize, we have $n$ different power generators, with $g_i$ power output from the $i^{th}$ generator in MW (decision variables), and $c^g_i$  as the incremental cost (\$ per MWh) of running the generator. We also are able to inject a certain amount of wind power into the system: let $w$ be the wind power injected, and $c^w$  be the incremental cost (\$ per MWh) of this.

For constraints, we have minumum ($g^{min}_i$) and maximum ($g^{max}_i$) possible power outputs for each generator:

$$g^{min}_i \leq g_i \leq g^{max}_i$$.

We also have a constraint on windpower injection, where $w^f$ is the wind power forecasted for the time period in question:

$$0 \leq w \leq w_f$$

Finally, we must supply all forecasted demand $d_j$, such that the total power from wind injection and generation is equal to demand:

$$\sum_i g_i + w = d$$

We can formulated this problem mathematically as follows to minimize costs of energy distribution:

$$
\begin{aligned}
& \underset{g_i \in I, w}{\text{minimize}}
& & Z = \sum_{i \in I} c^g_i g_i  + c^w w \\
& \text{subject to}
& & g^{min}_i \leq g_i \leq g^{max}_i \: \forall \: i\\
& & & 0 \leq w \leq w_f \\
& & & \sum_i g_i + w = d
\end{aligned}
$$

### Unit Commitment

Adding binary variables representing whether generators are on or off leads to the slightly more sophisticated Unit Commitment (UC) model. Here we introduce a binary variable, $u_i$, indicating whether each generator is on (1) or off (0). We obviously get no power from a generator that is off, but we also don't have any costs from that generator either. This changes our problem into a mixed integer linear optimization problem, by modifying the first set of constraints $ g^{min}_i \leq g_i \leq g^{max}_i$ to the be following constraints, for each $i$:

$$g^{min}_i u_i \leq g_i \leq g^{max}_i u_i \: \forall \: i$$

Note that this constraint lets - and also forces - $g_i$ be zero if the generator is off ($u_i=0$). Let's add this to the model and resolve.



First, install Pyomo and glpk if you are using Google Colab:

In [ ]:
!pip install -q pyomo
!apt-get install -y -qq glpk-utils

Selecting previously unselected package libsuitesparseconfig5:amd64.
(Reading database ... 123633 files and directories currently installed.)
Preparing to unpack .../libsuitesparseconfig5_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libamd2:amd64.
Preparing to unpack .../libamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libcolamd2:amd64.
Preparing to unpack .../libcolamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libcolamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libglpk40:amd64.
Preparing to unpack .../libglpk40_5.0-1_amd64.deb ...
Unpacking libglpk40:amd64 (5.0-1) ...
Selecting previously unselected package glpk-utils.
Preparing to unpack .../glpk-utils_5.0-1_amd64.deb ...
Unpacking glpk-utils (5.0-1) ...
Setting up libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4b

Next, reload the data:

In [ ]:
import pandas as pd
generator_data = pd.read_csv('generators.csv')
generator_data.head()

,Generator,Variable Cost $/MWh,Fixed Cost $,Min,Max
0,1,50,1000,0,1000
1,2,100,0,300,1000
2,3,75,100,0,900
3,4,65,100,5,1200


As before, let's load this generator data into variables anddefine variables for supplmentary data on our demand forecast, wind forecast, and cost of injecting wind.

In [ ]:
c_g = generator_data["Variable Cost $/MWh"].values
g_min = generator_data["Min"].values
g_max = generator_data["Max"].values
print("Wind table variable and column count:", generator_data.shape)

# demand forcast
demand = 1500.00
# wind forecast
w_forecast = 200.0
# wind cost
w_cost = 50.0


Wind table variable and column count: (4, 5)


First, let's re-solve the original economic dispatch model, for reference:



In [ ]:
from pyomo.environ import *

I = range(generator_data.shape[0])

# create a model
model = ConcreteModel()
#declare decision variables
model.g = Var(I,domain=NonNegativeReals)
model.w =  Var([1],domain=NonNegativeReals)
#declare objective
model.cost = Objective(expr = sum([c_g[i]*model.g[i] for i in I]) + w_cost*model.w[1], sense=minimize)
#declare constraints
model.cons = ConstraintList()
for i in I:
  model.cons.add(g_min[i] <= model.g[i])
  model.cons.add(model.g[i] <= g_max[i] )
model.cons.add(model.w[1] <= w_forecast)
model.cons.add(sum([model.g[i] for i in I]) + model.w[1] == demand)
#solve linear program
SolverFactory('glpk', executable='/usr/bin/glpsol').solve(model).write()
#display solution
print('\nCost = ', model.cost())
print('\nDecision Variables')
print('g = ', [model.g[i].value for i in I])
print('w = ', model.w[1].value)
print('Wind spill = ', w_forecast  - model.w[1].value)

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Name: unknown
  Lower bound: 90075.0
  Upper bound: 90075.0
  Number of objectives: 1
  Number of constraints: 10
  Number of variables: 5
  Number of nonzeros: 14
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 0
      Number of created subproblems: 0
  Error rc: 0
  Time: 0.004027605056762695
# ----------------------------------------------------------
#   Solution Information
# ------------------------------

Now, let's formulate this optimization problem in Pyomo. Our Objective function to minimize, $\sum_{i \in I} c^g_i g_i  + c^w w$, stays the same as in the Economic Dispatch problem. We add $u_i$ (for each $i$ ) as a binary variable, by adding the Pyomo variable `model.u = Var(I,domain=Binary)`.

Also, we then modify our constraint `model.cons.add(g_min[i]<= model.g[i]) ` to `model.cons.add(g_min[i]*model.u[i] <= model.g[i]) ` to include the binary variable, and likewise change  `model.cons.add(model.g[i] <= g_max[i] )` to ` model.cons.add(model.g[i] <= g_max[i]*model.u[i] )`, for each $i$. The rest of the model is the same as before. Run the code below to solve the model.

In [ ]:
from pyomo.environ import *

I = range(generator_data.shape[0])

# create a model
model = ConcreteModel()
#declare decision variables
model.g = Var(I,domain=NonNegativeReals)
model.w =  Var([1],domain=NonNegativeReals)
model.u = Var(I,domain=Binary) #NEW VARIABLE!
#declare objective
model.cost = Objective(expr = sum([c_g[i]*model.g[i] for i in I]) + w_cost*model.w[1], sense=minimize)
#declare constraints
model.cons = ConstraintList()
for i in I:
  model.cons.add(g_min[i]*model.u[i] <= model.g[i])  #MODIFIED CONSTRAINT!
  model.cons.add(model.g[i] <= g_max[i]*model.u[i] )  #MODIFIED CONSTRAINT!
model.cons.add(model.w[1] <= w_forecast)
model.cons.add(sum([model.g[i] for i in I]) + model.w[1] == demand)
#solve linear program
SolverFactory('glpk', executable='/usr/bin/glpsol').solve(model).write()
#display solution
print('\nCost = ', model.cost())
print('\nDecision Variables')
print('g = ', [model.g[i].value for i in I])
print('u = ', [model.u[i].value for i in I]) #PRINT OPTIMAL BINARY VARIABLES
print('w = ', model.w[1].value)
print('Wind spill = ', w_forecast  - model.w[1].value)

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Name: unknown
  Lower bound: 79500.0
  Upper bound: 79500.0
  Number of objectives: 1
  Number of constraints: 10
  Number of variables: 9
  Number of nonzeros: 20
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 1
      Number of created subproblems: 1
  Error rc: 0
  Time: 0.004683971405029297
# ----------------------------------------------------------
#   Solution Information
# ------------------------------

We now see that we should turn of Generators 2 and 3. This improves our cost from 90075 (the optimal value from the economic dispatch model) to 79500.
